In [7]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

# 1. Charger le dataset (Assure-toi que le nom du fichier est correct)
# On suppose ici que tu as gardé le fichier de l'étape EDA
columns = (['duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment','urgent','hot',
            'num_failed_logins','logged_in','num_compromised','root_shell','su_attempted','num_root','num_file_creations',
            'num_shells','num_access_files','num_outbound_cmds','is_host_login','is_guest_login','count','srv_count',
            'serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate','srv_diff_host_rate',
            'dst_host_count','dst_host_srv_count','dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
            'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate','dst_host_rerror_rate',
            'dst_host_srv_rerror_rate','attack_type','level'])

df = pd.read_csv('../data/KDDTrain+.txt', names=columns)
# Afficher les premières lignes pour vérifier
print("Dataset chargé avec succès !")
print(df.head())

# 2. Séparer les colonnes texte (catégorielles) et numériques
categorical_cols = ['protocol_type', 'service', 'flag']

# 3. Transformer le texte en nombres (Label Encoding)
encoder = LabelEncoder()
for col in categorical_cols:
    df[col] = encoder.fit_transform(df[col])

print("\n--- Après encodage du texte ---")
print(df[['protocol_type', 'service', 'flag']].head())


from sklearn.preprocessing import StandardScaler

# 4. Simplifier la cible (Classification Binaire : Normal = 0, Attaque = 1)
df['label'] = df['attack_type'].apply(lambda x: 0 if x == 'normal' else 1)

# On supprime l'ancienne colonne d'attaque détaillée et la colonne 'level' (qui ne sert pas à l'IA)
df = df.drop(['attack_type', 'level'], axis=1)

print("--- Répartition Normal (0) vs Attaques (1) ---")
print(df['label'].value_counts())

# 5. Séparer les caractéristiques (X) de la réponse attendue (y)
X = df.drop('label', axis=1)
y = df['label']

# 6. Normalisation (Mettre toutes les valeurs à la même échelle)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Reconvertir en DataFrame pour que ce soit lisible
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("\n--- Données prêtes pour l'IA (Aperçu) ---")
print(X_scaled_df.head())


import joblib
import os

# 1. Sauvegarder les données nettoyées dans le dossier data
X_scaled_df.to_csv('../data/KDDTrain_cleaned.csv', index=False)
pd.DataFrame(y, columns=['label']).to_csv('../data/KDDTrain_labels.csv', index=False)

# 2. Créer le dossier models s'il n'existe pas
os.makedirs('../models', exist_ok=True)

# 3. Sauvegarder l'outil de normalisation (Très important pour le Backend FastAPI plus tard)
joblib.dump(scaler, '../models/scaler.pkl')

print("✅ Données propres et Scaler sauvegardés avec succès ! Tâche 7 terminée.")

Dataset chargé avec succès !
   duration protocol_type   service flag  src_bytes  dst_bytes  land  \
0         0           tcp  ftp_data   SF        491          0     0   
1         0           udp     other   SF        146          0     0   
2         0           tcp   private   S0          0          0     0   
3         0           tcp      http   SF        232       8153     0   
4         0           tcp      http   SF        199        420     0   

   wrong_fragment  urgent  hot  ...  dst_host_same_srv_rate  \
0               0       0    0  ...                    0.17   
1               0       0    0  ...                    0.00   
2               0       0    0  ...                    0.10   
3               0       0    0  ...                    1.00   
4               0       0    0  ...                    1.00   

   dst_host_diff_srv_rate  dst_host_same_src_port_rate  \
0                    0.03                         0.17   
1                    0.60                  